# 02 — Treinamento YOLOv8 para Detecção de Violência

Este notebook treina o **YOLOv8m** (médio) com fine-tuning no dataset preparado no notebook anterior.

**Fluxo:**
1. Configurar hiperparâmetros
2. Verificar dataset e GPU
3. Iniciar treinamento com monitoramento inline
4. Visualizar curvas de loss / métricas
5. Avaliar no conjunto de teste
6. Exportar para ONNX (inferência rápida no OpenShift)

---
> **Dica de hardware:** Para 100 épocas com YOLOv8m e batch=16,  
> uma GPU A10G (OpenShift AI) leva ~4-6h. Use `yolov8n` para experimentos rápidos.

## 1. Instalação

In [ ]:
!pip install -q ultralytics PyYAML matplotlib seaborn

## 2. Imports e verificação de GPU

In [ ]:
import os
import yaml
import json
import shutil
from pathlib import Path
from datetime import datetime

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import pandas as pd
from ultralytics import YOLO
from IPython.display import display, Image as IPImage

# ── GPU / Device ───────────────────────────────────────────────────────────────
DEVICE = "0" if torch.cuda.is_available() else "cpu"

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Device   : {DEVICE}")

## 3. Configuração de hiperparâmetros

In [ ]:
# ── Caminhos ───────────────────────────────────────────────────────────────────
DATASET_YAML  = Path("../data/processed/dataset.yaml")
MODELS_DIR    = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Hiperparâmetros ────────────────────────────────────────────────────────────
CFG = {
    # Modelo base — troque por 'yolov8n.pt' para experimentos rápidos
    "model":         "yolov8m.pt",

    "epochs":        100,
    "batch":         16,
    "img_size":      640,
    "device":        DEVICE,

    # Learning rate
    "lr0":           0.001,
    "lrf":           0.01,      # lr final = lr0 * lrf
    "momentum":      0.937,
    "weight_decay":  0.0005,
    "warmup_epochs": 3,
    "patience":      20,        # early stopping
    "optimizer":     "AdamW",

    # Loss — cls aumentado para dar mais peso às classes violence/pre_violence
    "cls":  0.7,
    "box":  7.5,
    "dfl":  1.5,

    # Augmentação agressiva para generalização
    "augment":     True,
    "mosaic":      1.0,
    "mixup":       0.15,
    "copy_paste":  0.10,
    "degrees":     10.0,
    "translate":   0.1,
    "scale":       0.5,
    "fliplr":      0.5,
    "hsv_h":       0.015,
    "hsv_s":       0.7,
    "hsv_v":       0.4,

    # Saída
    "save_period": 10,
}

RUN_NAME = f"violence_yolov8_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Run name : {RUN_NAME}")
print(f"Dataset  : {DATASET_YAML.resolve()}")
print(f"Epochs   : {CFG['epochs']}   Batch: {CFG['batch']}   Img: {CFG['img_size']}")

In [ ]:
# ── Verificar dataset.yaml ────────────────────────────────────────────────────
assert DATASET_YAML.exists(), (
    f"dataset.yaml não encontrado em {DATASET_YAML}\n"
    "Execute o notebook 01_dataset_preparation.ipynb primeiro."
)

with open(DATASET_YAML) as f:
    ds_cfg = yaml.safe_load(f)

print("dataset.yaml:")
for k, v in ds_cfg.items():
    print(f"  {k}: {v}")

# Contar imagens por split
ds_path = Path(ds_cfg["path"])
for split in ["train", "val", "test"]:
    n = len(list((ds_path / "images" / split).glob("*.jpg")))
    print(f"  {split}: {n} imagens")

## 4. Treinamento

In [ ]:
# ── Carregar modelo base (download automático do COCO pretrained) ──────────────
model = YOLO(CFG["model"])
print(f"Modelo carregado: {CFG['model']}")
print(f"Parâmetros: {sum(p.numel() for p in model.model.parameters()):,}")

In [ ]:
# ── Treinamento ────────────────────────────────────────────────────────────────
# O Ultralytics exibe curvas de loss em tempo real no terminal.
# Ao final, os pesos são salvos em MODELS_DIR/RUN_NAME/yolo/weights/

results = model.train(
    data          = str(DATASET_YAML),
    epochs        = CFG["epochs"],
    imgsz         = CFG["img_size"],
    batch         = CFG["batch"],
    device        = CFG["device"],
    # Learning rate
    lr0           = CFG["lr0"],
    lrf           = CFG["lrf"],
    momentum      = CFG["momentum"],
    weight_decay  = CFG["weight_decay"],
    warmup_epochs = CFG["warmup_epochs"],
    patience      = CFG["patience"],
    optimizer     = CFG["optimizer"],
    # Loss
    cls           = CFG["cls"],
    box           = CFG["box"],
    dfl           = CFG["dfl"],
    # Augmentação
    augment       = CFG["augment"],
    mosaic        = CFG["mosaic"],
    mixup         = CFG["mixup"],
    copy_paste    = CFG["copy_paste"],
    degrees       = CFG["degrees"],
    translate     = CFG["translate"],
    scale         = CFG["scale"],
    fliplr        = CFG["fliplr"],
    hsv_h         = CFG["hsv_h"],
    hsv_s         = CFG["hsv_s"],
    hsv_v         = CFG["hsv_v"],
    # Saída
    project       = str(MODELS_DIR / RUN_NAME),
    name          = "yolo",
    save_period   = CFG["save_period"],
    verbose       = True,
)

print("\n✓ Treinamento concluído")

## 5. Visualização das métricas de treinamento

In [ ]:
# ── Carregar results.csv gerado pelo Ultralytics ───────────────────────────────
run_dir     = MODELS_DIR / RUN_NAME / "yolo"
results_csv = run_dir / "results.csv"

assert results_csv.exists(), f"results.csv não encontrado em {results_csv}"

df = pd.read_csv(results_csv)
df.columns = [c.strip() for c in df.columns]
print("Colunas disponíveis:")
print(df.columns.tolist())
df.tail(5)

In [ ]:
# ── Curvas de loss e métricas ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

def plot_col(ax, col, title, color="steelblue"):
    if col in df.columns:
        ax.plot(df["epoch"], df[col], color=color, lw=2)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Época")
        ax.grid(alpha=0.3)
    else:
        ax.set_title(f"{title} (N/A)")

plot_col(axes[0, 0], "train/box_loss",   "Train Box Loss",     "tomato")
plot_col(axes[0, 1], "train/cls_loss",   "Train Cls Loss",     "darkorange")
plot_col(axes[0, 2], "train/dfl_loss",   "Train DFL Loss",     "goldenrod")
plot_col(axes[1, 0], "val/box_loss",     "Val Box Loss",       "steelblue")
plot_col(axes[1, 1], "metrics/mAP50(B)", "mAP@50",             "seagreen")
plot_col(axes[1, 2], "metrics/mAP50-95(B)", "mAP@50-95",      "mediumslateblue")

plt.suptitle(f"Métricas de Treinamento — {RUN_NAME}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Precision / Recall por época ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_col(axes[0], "metrics/precision(B)", "Precision", "steelblue")
plot_col(axes[1], "metrics/recall(B)",    "Recall",    "tomato")
plt.tight_layout()
plt.show()

# Melhor mAP50
if "metrics/mAP50(B)" in df.columns:
    best_epoch = df["metrics/mAP50(B)"].idxmax()
    best_map   = df.loc[best_epoch, "metrics/mAP50(B)"]
    print(f"\nMelhor mAP@50 : {best_map:.4f}  (época {int(df.loc[best_epoch, 'epoch'])})")

In [ ]:
# ── Matriz de confusão gerada pelo Ultralytics ────────────────────────────────
confusion_img = run_dir / "confusion_matrix_normalized.png"
if confusion_img.exists():
    img = mpimg.imread(str(confusion_img))
    plt.figure(figsize=(7, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Matriz de Confusão (normalizada)", fontsize=12)
    plt.show()
else:
    print("Matriz de confusão ainda não gerada.")

## 6. Avaliação no conjunto de teste

In [ ]:
# ── Carregar o melhor checkpoint ──────────────────────────────────────────────
best_weights = run_dir / "weights" / "best.pt"
assert best_weights.exists(), f"Pesos não encontrados: {best_weights}"

best_model = YOLO(str(best_weights))
print(f"Pesos carregados: {best_weights}")

In [ ]:
# ── Validação no conjunto de teste ────────────────────────────────────────────
test_metrics = best_model.val(
    data   = str(DATASET_YAML),
    split  = "test",
    imgsz  = CFG["img_size"],
    device = CFG["device"],
)

print("\n── Métricas no Teste ───────────────────────────")
print(f"mAP@50      : {test_metrics.box.map50:.4f}")
print(f"mAP@50-95   : {test_metrics.box.map:.4f}")
print(f"Precision   : {test_metrics.box.mp:.4f}")
print(f"Recall      : {test_metrics.box.mr:.4f}")

# Por classe
class_names = ["person", "violence", "pre_violence"]
if hasattr(test_metrics.box, "ap_class_index"):
    print("\nPor classe:")
    for i, ap50 in enumerate(test_metrics.box.ap50):
        print(f"  {class_names[i]:14s}  AP@50={ap50:.4f}")

In [ ]:
# ── Visualizar predições em imagens de teste ──────────────────────────────────
test_images = list((Path(ds_cfg["path"]) / "images" / "test").glob("*.jpg"))[:6]
CLASS_COLORS_BGR = {0: (0,200,0), 1: (0,0,255), 2: (0,140,255)}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, img_path in zip(axes.flat, test_images):
    frame   = __import__("cv2").imread(str(img_path))
    preds   = best_model(frame, verbose=False, conf=0.3)[0]
    vis     = frame.copy()

    for box in preds.boxes:
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = (int(v) for v in box.xyxy[0].tolist())
        col = CLASS_COLORS_BGR.get(cls, (128,128,128))
        __import__("cv2").rectangle(vis, (x1,y1), (x2,y2), col, 2)
        __import__("cv2").putText(vis, f"{class_names[cls]} {conf:.2f}",
                                  (x1, y1-5), __import__("cv2").FONT_HERSHEY_SIMPLEX,
                                  0.5, col, 1)

    ax.imshow(__import__("cv2").cvtColor(vis, __import__("cv2").COLOR_BGR2RGB))
    ax.axis("off")

plt.suptitle("Predições no conjunto de teste", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Exportar para ONNX (inferência otimizada no OpenShift)

In [ ]:
# ── Exportar ONNX ─────────────────────────────────────────────────────────────
print("Exportando modelo para ONNX...")
export_path = best_model.export(
    format   = "onnx",
    imgsz    = CFG["img_size"],
    simplify = True,
    dynamic  = False,
    opset    = 17,
)
print(f"✓ ONNX salvo em: {export_path}")

# Copiar pesos finais para pasta central
shutil.copy(str(best_weights), str(MODELS_DIR / "best.pt"))
print(f"✓ best.pt copiado para: {MODELS_DIR / 'best.pt'}")

In [ ]:
# ── Resumo final ──────────────────────────────────────────────────────────────
print("=" * 50)
print(" RESUMO DO TREINAMENTO")
print("=" * 50)
print(f" Run          : {RUN_NAME}")
print(f" Best weights : {MODELS_DIR / 'best.pt'}")
print(f" ONNX export  : {export_path}")
print(f" mAP@50 (test): {test_metrics.box.map50:.4f}")
print("=" * 50)
print(" Próximo passo: 03_lstm_pre_violence.ipynb")